# Client maker
<br>This scipt makes the csv of the clients with their preferences and address. 
<br>The addresses need to be calculated first in the data_generation/address_generator.ipynb file.

In [11]:
import random
import csv
from pathlib import Path
import osmnx as ox
import pandas as pd

In [12]:
heerlen_addresses_df = pd.read_csv("../output/heerlen_addresses.csv")

In [13]:
# Use prevalidated addresses from heerlen_addresses_df.
required_columns = {"full_address", "coordinates"}
missing_columns = required_columns - set(heerlen_addresses_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns in heerlen_addresses_df: {sorted(missing_columns)}")

VALID_HEERLEN_ADDRESS_ROWS = (
    heerlen_addresses_df[["full_address", "coordinates"]]
    .dropna(subset=["full_address"])
    .assign(full_address=lambda d: d["full_address"].astype(str).str.strip())
    .drop_duplicates(subset=["full_address"])
    .to_dict("records")
)

if not VALID_HEERLEN_ADDRESS_ROWS:
    raise ValueError("No usable rows found in heerlen_addresses_df['full_address'].")

# Shuffle rows and cycle through them to avoid duplicate addresses until all are used.
random.shuffle(VALID_HEERLEN_ADDRESS_ROWS)
_ADDRESS_INDEX = 0

WORK_DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday"]
WORK_DAY_ORDER = {day: idx for idx, day in enumerate(WORK_DAYS)}

LAST_NAMES_NL = [
    "de Vries", "Jansen", "de Jong", "Bakker", "Visser", "Smit", "Meijer", "Mulder", "Bos", "Dekker",
    "Peters", "Hendriks", "van Dijk", "van den Berg", "van Leeuwen", "Kuipers", "Jacobs", "de Boer", "Vos", "Schouten",
    "Willems", "Hermans", "Kok", "Vermeulen", "de Graaf", "Martens", "van Loon", "van der Meer", "Kramer", "Postma",
]

CARE_WINDOW_START_MIN = 14 * 60
CARE_WINDOW_END_MIN = 18 * 60
CARE_TIME_STEP_MIN = 30
MIN_PADDING_MIN = 30
MAX_PADDING_MIN = 120

def _minutes_to_hhmm(total_minutes: int) -> str:
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours:02d}:{minutes:02d}"

def _generate_care_window(care_hours):
    """Generate a care-availability window between 14:00 and 18:00 with padding around care time."""
    care_duration_min = int(round(float(care_hours) * 60))
    full_range_min = CARE_WINDOW_END_MIN - CARE_WINDOW_START_MIN

    min_window_duration = min(full_range_min, care_duration_min + MIN_PADDING_MIN)
    max_window_duration = min(full_range_min, care_duration_min + MAX_PADDING_MIN)

    if min_window_duration > max_window_duration:
        min_window_duration = max_window_duration

    window_duration_min = random.randrange(
        min_window_duration,
        max_window_duration + CARE_TIME_STEP_MIN,
        CARE_TIME_STEP_MIN,
    )

    latest_start = CARE_WINDOW_END_MIN - window_duration_min
    window_start_min = random.randrange(
        CARE_WINDOW_START_MIN,
        latest_start + CARE_TIME_STEP_MIN,
        CARE_TIME_STEP_MIN,
    )
    window_end_min = window_start_min + window_duration_min

    return _minutes_to_hhmm(window_start_min), _minutes_to_hhmm(window_end_min)

def _generate_client_name(index):
    sex = random.choice(["man", "woman"])
    salutation = "Dhr." if sex == "man" else "Mw."
    last_name = random.choice(LAST_NAMES_NL)
    # Keep names unique by appending index while preserving requested salutation/last name style.
    return f"{salutation} {last_name} {index}"

def generate_real_address(max_attempts: int = 1):
    """Return one known Heerlen address row with its coordinates."""
    global _ADDRESS_INDEX
    row = VALID_HEERLEN_ADDRESS_ROWS[_ADDRESS_INDEX % len(VALID_HEERLEN_ADDRESS_ROWS)]
    _ADDRESS_INDEX += 1
    return row

# Possible care arrangements
CARE_ARRANGEMENTS = ["HBH Basic", "HBH Plus", "Wash & Ironing", "V&V"]
CARE_HOURS = [1, 1.5, 2, 2.5, 3]   # step 0.5

def generate_client(index):
    """Generate a single client record as a dictionary."""
    name = _generate_client_name(index)
    address_row = generate_real_address()
    address = address_row["full_address"]
    coordinates = address_row.get("coordinates")
    care_arrangement = random.choice(CARE_ARRANGEMENTS)

    care_hours = random.choice(CARE_HOURS)
    time_window_start, time_window_end = _generate_care_window(care_hours)

    available_days = random.sample(WORK_DAYS, k=random.choice([4, 5]))
    available_days = sorted(available_days, key=lambda d: WORK_DAY_ORDER[d])
    availability = ",".join(available_days)

    # Most clients have no pets; occasional 1 or 2
    dogs = random.choices([0, 1, 2], weights=[0.7, 0.2, 0.1])[0]
    cats = random.choices([0, 1, 2], weights=[0.6, 0.3, 0.1])[0]

    # 30% chance of smoking
    smokes = random.random() < 0.3

    return {
        "name": name,
        "address": address,
        "coordinates": coordinates,
        "care_arrangement": care_arrangement,
        "availability": availability,
        "time_window_start": time_window_start,
        "time_window_end": time_window_end,
        "care_hours": care_hours,
        "dogs": dogs,
        "cats": cats,
        "smokes": smokes,
    }

def generate_clients_csv(num_clients, filename="../output/clients.csv"):
    """Generate num_clients records and write them to a CSV file."""
    fieldnames = [
        "name", "address", "coordinates", "care_arrangement", "availability",
        "time_window_start", "time_window_end", "care_hours",
        "dogs", "cats", "smokes"
    ]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for i in range(num_clients):
            client = generate_client(i)
            # Convert boolean to lowercase string for readability
            client["smokes"] = str(client["smokes"]).lower()
            writer.writerow(client)

    print(f"Generated {num_clients} clients in '{output_path}'.")

if __name__ == "__main__":
    # Generate 100 clients by default
    generate_clients_csv(100)

Generated 100 clients in '..\output\clients.csv'.
